# Assignment 07: NLP CV Analyzer

**Course Project:** Natural Language Processing (NLP)  
**Project:** NLP-based CV/Resume Analyzer

### Objective
This project demonstrates NLP techniques to analyze a CV/resume against a selected job role. It extracts resume text, preprocesses it, calculates an ATS-style match score using **TF-IDF + cosine similarity**, identifies matched and missing skills, and generates resume suggestions.

### Main Features
1. Upload/read a Resume (PDF/DOCX/TXT)
2. Select a target job role
3. Extract and preprocess CV text
4. Calculate an ATS-style match score
5. Identify matched skills
6. Identify missing skills
7. Generate resume suggestions
8. Display a simple visualization

> This notebook follows the same overall project idea as the provided Streamlit application: resume upload, job selection, ATS score, matched/missing skills, strengths, suggestions, and extracted-resume preview.

## 1. Install / Import Libraries

In [ ]:
# If needed, uncomment this line in a fresh environment:
# !pip install nltk pdfplumber python-docx scikit-learn pandas numpy matplotlib wordcloud

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Libraries imported successfully.")

## 2. Sample Job Database

In [ ]:
JOB_DATABASE = {
    "AI Engineer": {
        "description": (
            "Develop AI and machine learning applications using Python, NLP, "
            "machine learning, deep learning, APIs, data analysis and generative AI. "
            "Build intelligent automation and deploy AI solutions."
        ),
        "skills": [
            "Python", "Machine Learning", "Deep Learning", "NLP",
            "Generative AI", "APIs", "Data Analysis", "SQL", "Git"
        ]
    },
    "Social Media Marketing Manager": {
        "description": (
            "Plan and execute social media strategies, create content, manage "
            "Instagram, Facebook, TikTok and LinkedIn, run digital advertising, "
            "generate leads, analyze campaigns and improve engagement."
        ),
        "skills": [
            "Social Media Marketing", "Content Creation", "Video Editing",
            "Meta Ads", "Google Ads", "Lead Generation", "Instagram",
            "Facebook", "TikTok", "LinkedIn", "Analytics"
        ]
    },
    "Data Analyst": {
        "description": (
            "Analyze business data using Python, pandas, NumPy, SQL and statistics. "
            "Create reports, dashboards and visualizations and communicate insights."
        ),
        "skills": [
            "Python", "Pandas", "NumPy", "SQL", "Statistics",
            "Data Analysis", "Data Visualization", "Excel", "Power BI"
        ]
    },
    "Customer Support Representative": {
        "description": (
            "Handle customer queries through chat, email and calls, resolve issues, "
            "maintain customer satisfaction, communicate professionally and manage CRM records."
        ),
        "skills": [
            "Customer Support", "Communication", "CRM", "Chat Support",
            "Problem Solving", "Email Support", "Call Handling"
        ]
    }
}

print("Available job roles:")
for role in JOB_DATABASE:
    print("-", role)

## 3. Resume Text

In [ ]:
# For this assignment demo, a sample CV is included.
# You can replace SAMPLE_RESUME with the text extracted from your own PDF/DOCX.

SAMPLE_RESUME = """
Muhammad Abdullah Nasir

Professional Summary
AI and Digital Marketing professional with experience in social media marketing,
content creation, video editing, lead generation and customer support. Interested
in AI engineering, generative AI and NLP.

Education
BS Information Technology

Experience
Social Media Manager
Planned social media campaigns, created short-form video content, managed Instagram,
Facebook and TikTok pages, generated leads and improved audience engagement.

Customer Support Representative
Handled customer queries through chat and calls, resolved customer issues and
maintained professional communication.

Technical Skills
Python, NLP, Machine Learning, Generative AI, Prompt Engineering, Social Media Marketing,
Content Creation, Video Editing, Meta Ads, Lead Generation, Instagram, Facebook,
TikTok, LinkedIn, Microsoft Office, Communication, Problem Solving.
"""

print(SAMPLE_RESUME[:1000])

## 4. Resume Text Extraction Functions

In [ ]:
def extract_text_from_pdf(file_path):
    import pdfplumber
    text = []
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text.append(page_text)
    return "\n".join(text)

def extract_text_from_docx(file_path):
    from docx import Document
    doc = Document(file_path)
    return "\n".join(p.text for p in doc.paragraphs)

def extract_resume_text(file_path):
    """Extract text from PDF, DOCX or TXT."""
    suffix = Path(file_path).suffix.lower()

    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)
    elif suffix == ".docx":
        return extract_text_from_docx(file_path)
    elif suffix == ".txt":
        return Path(file_path).read_text(encoding="utf-8")
    else:
        raise ValueError("Supported formats: PDF, DOCX, TXT")

## 5. NLP Preprocessing

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"\S+@\S+", " ", text)          # remove emails
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"\d+", " ", text)               # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))

    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return " ".join(tokens)

clean_resume = preprocess_text(SAMPLE_RESUME)

print("Original words:", len(SAMPLE_RESUME.split()))
print("Processed words:", len(clean_resume.split()))
print("\nProcessed text:\n")
print(clean_resume[:1000])

## 6. Select Target Job

In [ ]:
selected_job = "AI Engineer"

job_description = JOB_DATABASE[selected_job]["description"]
required_skills = JOB_DATABASE[selected_job]["skills"]

print("Target Job:", selected_job)
print("Required Skills:", ", ".join(required_skills))

## 7. ATS Match Score using TF-IDF + Cosine Similarity

**TF-IDF** converts text into numerical feature vectors based on word importance.

**Cosine similarity** measures how similar the resume and job description are.

Formula:

**Cosine Similarity = (A · B) / (||A|| × ||B||)**

The similarity value is converted into a percentage for an ATS-style score.

In [ ]:
clean_job = preprocess_text(job_description)

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform([clean_resume, clean_job])

similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
ats_score = round(similarity * 100, 2)

print(f"ATS Match Score: {ats_score}%")

## 8. Skill Matching

In [ ]:
def normalize_skill(skill):
    return re.sub(r"[^a-z0-9+#. ]", "", skill.lower()).strip()

resume_lower = SAMPLE_RESUME.lower()

matched_skills = []
missing_skills = []

for skill in required_skills:
    if normalize_skill(skill) in resume_lower:
        matched_skills.append(skill)
    else:
        missing_skills.append(skill)

skill_match_percentage = round(
    len(matched_skills) / len(required_skills) * 100, 2
)

print("Matched Skills:")
for skill in matched_skills:
    print("✓", skill)

print("\nMissing Skills:")
for skill in missing_skills:
    print("✗", skill)

print(f"\nSkill Coverage: {skill_match_percentage}%")

## 9. Resume Strengths and Suggestions

In [ ]:
def generate_strengths(score, matched, resume_text):
    strengths = []

    if score >= 70:
        strengths.append("Strong similarity with the selected job description.")
    elif score >= 50:
        strengths.append("Moderate similarity with the selected job description.")
    else:
        strengths.append("The resume needs stronger alignment with the target role.")

    if matched:
        strengths.append(f"{len(matched)} required job skills were found in the resume.")

    if any(word in resume_text.lower() for word in ["experience", "work", "internship"]):
        strengths.append("The resume contains professional/experience-related information.")

    if any(word in resume_text.lower() for word in ["python", "machine learning", "nlp"]):
        strengths.append("The resume contains relevant technical keywords.")

    return strengths

def generate_suggestions(missing, score):
    suggestions = []

    if missing:
        suggestions.append(
            "Add relevant missing skills only if you genuinely have those skills."
        )
        suggestions.append(
            "Use job-specific keywords naturally in the Skills and Experience sections."
        )

    if score < 60:
        suggestions.append(
            "Rewrite the professional summary to match the target job more closely."
        )

    suggestions.append(
        "Add measurable achievements where possible, such as percentages, revenue, "
        "lead counts, engagement growth or project outcomes."
    )

    return suggestions

strengths = generate_strengths(ats_score, matched_skills, SAMPLE_RESUME)
suggestions = generate_suggestions(missing_skills, ats_score)

print("Resume Strengths")
for item in strengths:
    print("•", item)

print("\nSuggestions")
for item in suggestions:
    print("•", item)

## 10. Keyword Frequency Analysis

In [ ]:
words = preprocess_text(SAMPLE_RESUME).split()
word_counts = Counter(words)

top_words = word_counts.most_common(15)
keywords_df = pd.DataFrame(top_words, columns=["Keyword", "Frequency"])

display(keywords_df)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(keywords_df["Keyword"], keywords_df["Frequency"])
plt.xticks(rotation=45, ha="right")
plt.title("Top Resume Keywords")
plt.xlabel("Keyword")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 11. Final Analysis Dashboard

In [ ]:
analysis_summary = pd.DataFrame({
    "Metric": [
        "ATS Match Score",
        "Skill Coverage",
        "Matched Skills",
        "Missing Skills"
    ],
    "Result": [
        f"{ats_score}%",
        f"{skill_match_percentage}%",
        len(matched_skills),
        len(missing_skills)
    ]
})

display(analysis_summary)

print("\nTarget Job:", selected_job)
print("\nMatched Skills:", ", ".join(matched_skills) if matched_skills else "None")
print("Missing Skills:", ", ".join(missing_skills) if missing_skills else "None")

if ats_score >= 80:
    print("\nStatus: Excellent Match")
elif ats_score >= 60:
    print("\nStatus: Good Match")
else:
    print("\nStatus: Needs Improvement")

## 12. Conclusion

The NLP CV Analyzer demonstrates how NLP can be applied to resume screening.

### NLP techniques used
- Text extraction from resume documents
- Text normalization
- Tokenization
- Stop-word removal
- Lemmatization
- TF-IDF vectorization
- Cosine similarity
- Keyword/skill matching
- Frequency analysis

### Result
The system provides an ATS-style score, matched skills, missing skills, resume strengths and suggestions. This can help a candidate identify gaps between a CV and a target job description.

**Note:** The ATS score is an educational similarity score, not a prediction of how a real company's ATS will rank a candidate.